# Does the risk equation work? — the composed score against a realised outcome

Every notebook before this one examined a **term**. `05-risk-composition.ipynb` showed which terms
have data behind them; `08-lsms-eda.ipynb` built exposure and vulnerability from survey data;
`09-croploss-model.ipynb` fitted the first model in this project to a **real** agronomic outcome.

None of them scored `Risk = Hazard x Exposure x Vulnerability` **as a whole** against anything that
actually happened. This notebook does, and it is the first time in the repository that the composed
number has been compared to an outcome.

**The join that makes it possible.** E07 composed risk per household from Nigeria GHS-Panel Wave 5.
E10 built a realised crop-loss label from the same survey, same wave. They share `hhid`. 2,875
households overlap.

**What it can and cannot settle.** E07's `hazard` is *drawn* from the panel's distribution -- Wave 5
ships no coordinates, so no household has a real hazard. So this validates the **exposure x
vulnerability** composition, which is 77.4% of `var(log expected_loss)` and the part actually under
dispute. It cannot validate the hazard model. Nothing here can, until outcome capture runs.

That drawn hazard is also a gift: it is a **negative control**, and section 2 uses it as the
validity gate before any result is read.

**Experiment:** `experiments/E11-composed-validation/`. This notebook reads that experiment's
committed result and re-derives its figures rather than restating them.

## 1. Setup

In [ ]:
import json, sys
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, '../experiments/E11-composed-validation')
import validate_composition as E

res = json.load(open('../experiments/E11-composed-validation/e11_result.json'))
df = E.load()

RED, BLUE, GREY = '#c0524a', '#3b6fa0', '#8a8a8a'
print(f"git sha {res['git_sha']} | seed {res['seed']}")
print(f"{res['n_households']} households, {res['n_ea']} EAs, "
      f"prevalence(any_loss) = {res['prevalence_any_loss']}")
print(f"population mean vulnerability = {res['mean_vulnerability_all']}")

## 2. The validity gate, before any result

`hazard` here is drawn, so it carries no information about who lost crop. If it scores above chance
against the realised label, the join, the label or the metric is wrong and nothing below may be
believed. This is the check the rest of the notebook rests on, so it comes first.

In [ ]:
ctrl = res['rankings']['hazard_only__NEGATIVE_CONTROL']
lo, hi = ctrl['auc_ci_ea_bootstrap']
print(f"drawn hazard alone: AUC {ctrl['auc']:.3f}  95% CI [{lo:.3f}, {hi:.3f}]")
print("PASS - interval spans 0.5" if lo <= 0.5 <= hi else "FAIL - stop reading")

## 3. The result

Unit is the **household**, because a visit goes to a farmer, not a plot. Intervals resample
**enumeration areas**, never households -- the same rule this repo applies to sites in
`04-leakage-and-validation.ipynb` ("39 monthly observations of one site are not 39 independent
facts").

The ceiling arm is E10's fitted model, out of fold, aggregated to the household. It is here for the
same reason `03` keeps climatology: a result without the baseline that could replace it is a
headline, not a finding.

In [ ]:
rows = []
for name, e in res['rankings'].items():
    lo, hi = e['auc_ci_ea_bootstrap']
    rows.append({'ranking': name, 'AUC': e['auc'], 'lo': lo, 'hi': hi,
                 'PR-AUC': e['pr_auc'], 'P@50': e['p_at_50'], 'P@200': e['p_at_200'],
                 'vuln@50': e['mean_vulnerability_top50']})
tbl = pd.DataFrame(rows).set_index('ranking')
display(tbl.round(4))
print(f"base rate (a random queue's P@k) = {res['prevalence_any_loss']}")

In [ ]:
t = tbl.sort_values('AUC')
fig, ax = plt.subplots(figsize=(9, 4.2))
colors = [RED if i == 'expected_loss' else
          ('#2e7d32' if 'CEILING' in i else (GREY if 'CONTROL' in i else BLUE)) for i in t.index]
for i, (name, r) in enumerate(t.iterrows()):
    ax.plot([r['lo'], r['hi']], [i, i], color=colors[i], lw=2, solid_capstyle='butt')
    ax.plot([r['lo'], r['lo'], np.nan, r['hi'], r['hi']],
            [i - .12, i + .12, np.nan, i - .12, i + .12], color=colors[i], lw=1.2)
ax.scatter(t['AUC'], range(len(t)), color=colors, zorder=3, s=55)
ax.axvline(0.5, color='k', ls='--', lw=1, label='chance')
ax.set_yticks(range(len(t))); ax.set_yticklabels(t.index, fontsize=9)
ax.set_xlabel('AUC against realised crop loss (95% CI, EA bootstrap)')
ax.set_title('The composed queue barely clears chance; a fitted model clears it comfortably')
ax.legend(loc='lower right', fontsize=8)
fig.tight_layout(); plt.show()

### Finding 1 — composing makes the ranking *worse* than its own exposure term

Comparing two independent intervals is the wrong test here: both are wide because EAs vary, and that
variance is common to both arms. The contrasts below are **paired** -- both rankings scored on the
same resampled EAs, so the shared noise cancels.

In [ ]:
con = pd.DataFrame(res['paired_contrasts']).T
con[['lo', 'hi']] = pd.DataFrame(con['ci'].tolist(), index=con.index)
display(con[['delta_auc', 'lo', 'hi', 'p_positive']].round(4))

fig, ax = plt.subplots(figsize=(9, 3.2))
y = range(len(con))
ax.errorbar(con['delta_auc'], y, xerr=[con['delta_auc'] - con['lo'], con['hi'] - con['delta_auc']],
            fmt='o', color='k', capsize=3, ls='none')
ax.axvline(0, color=RED, ls='--', lw=1)
ax.set_yticks(list(y)); ax.set_yticklabels(con.index, fontsize=9)
ax.set_xlabel('paired ΔAUC (same resampled EAs)')
ax.set_title('Multiplying exposure by the other two terms costs ordering information')
fig.tight_layout(); plt.show()

`expected_loss` scores **below** `exposure` alone, and the paired interval excludes zero. Multiplying
exposure by a drawn hazard and a real vulnerability does not add ordering information -- it removes
some. The composition is not merely exposure-dominated, as `docs/CONCEPTS.md` Rule 3 established; it
is exposure, degraded.

Two honest qualifications: the hazard being drawn is part of why the product loses information, and
a real hazard could change this. And the exposure term's own edge is small and partly a confound --
section 4.

## 4. The confound, checked rather than assumed

`any_loss` rises mechanically with plot count -- a household with more crop-plot rows has more
chances for one to fail -- and `value_at_risk_usd` is area-driven, so it correlates with plot count
too. Any edge exposure shows could be counting plots rather than finding risk.

Two ways to separate them, both reported: `share_loss`, a **rate** that count cannot inflate; and
AUC within fixed plot-count strata.

In [ ]:
cc = res['confound_check']
print(f"AUC, plot count alone : {cc['auc_plot_count_alone']:.4f}")
print(f"AUC, exposure alone   : {cc['auc_exposure_alone']:.4f}")
print(f"spearman(exposure, plot count) = {cc['spearman_exposure_vs_plot_count']:+.4f}\n")
print("on `share_loss`, which a plot count cannot inflate:")
for k, v in cc['spearman_vs_share_loss'].items():
    print(f"  spearman({k:14s}, share_loss) = {v:+.4f}")
print("\nAUC of exposure within fixed plot-count strata:")
for k, v in cc['auc_exposure_within_plot_count_strata'].items():
    print(f"  {k}: n={v['n']:4d}  prevalence={v['prevalence']:.3f}  AUC={v['auc_exposure']:.4f}")

Most of exposure's edge **is** the count: 0.553 from plot count alone against exposure's 0.564.
Within strata a small edge survives (0.521 / 0.534 / 0.549), and on the count-free `share_loss`
label exposure keeps **+0.117** spearman.

So larger farms really do lose crop slightly more often — exposure is not a pure magnitude term, and
that is a partial defence of its presence in the ranking. It is not a mandate to own 77% of it.

## 5. The robustness result: the queue does not survive its own measurement error

For a triage product, being wrong about a dollar figure matters far less than **the order moving**.
Every bias below is already measured in this repository, not invented:

* per-crop yield against FAO on sole-cropped plots — rice 0.87x, sorghum 0.84x, maize 0.44x,
  cassava 0.21x (`experiments/E07-lsms/FINDINGS.md`);
* GPS-measured plot area at 0.88x self-reported, **spearman +0.549**, n=5,137. The constant does not
  move a ranking; the *disagreement* does, so scenario B calibrates multiplicative noise to
  reproduce that rank correlation.

In [ ]:
st = {k: v for k, v in res['rank_stability'].items() if isinstance(v, dict)}
labels = {'A_yield_corrected_to_fao': 'A: yields corrected to FAO',
          'B_area_measurement_error': 'B: plot-area measurement error',
          'C_both': 'C: both'}
ks = [50, 100, 200]
fig, ax = plt.subplots(figsize=(8.5, 4))
w = 0.26
for i, (key, s) in enumerate(st.items()):
    vals = [s[f'top{k}_retained'] for k in ks]
    ax.bar([x + i * w for x in range(len(ks))], vals, w,
           label=f"{labels[key]}  (ρ={s['spearman_vs_base']})",
           color=[BLUE, RED, '#7a4b8a'][i])
    for x, v in zip(range(len(ks)), vals):
        ax.text(x + i * w, v + 0.015, f'{v:.2f}', ha='center', fontsize=8)
ax.axhline(1.0, color='k', ls=':', lw=1)
ax.set_xticks([x + w for x in range(len(ks))]); ax.set_xticklabels([f'top {k}' for k in ks])
ax.set_ylabel('share of the original top-k retained'); ax.set_ylim(0, 1.12)
ax.set_title('Under documented input error, only 10% of the top 50 survives')
ax.legend(fontsize=8, loc='upper left')
fig.tight_layout(); plt.show()
print(f"area-noise sigma calibrated to spearman {E.AREA_RANK_CORR}: "
      f"{res['rank_stability']['area_noise_sigma']}")

**This is the robustness answer, and it is independent of every other finding here.** The systematic
yield correction is benign (84% of the top 50 retained). It is the *random* area error that destroys
the order: **10%**.

Ranking farmers 1..50 asserts a precision the inputs cannot support. Either fix area measurement, or
ship *"these ~200 farmers, unordered"* — which is a real product change that costs nothing and stops
the service claiming precision it does not have.

## 6. `risk_score` cannot order a 50-farmer queue at all

`assess_risk` returns `round(score, 1)`. Across 2,875 households that leaves 557 distinct values, so
the 50th and 51st farmer are tied and the top 50 is an artifact of row order.

`model-design.md` §9.3a learned exactly this lesson — *"`combined` is stored at 6dp rather than 3;
at 3dp, rounding alone collapsed 140 distinct top-half values back to 8"* — and applied it to
`hazard.combined`. It was never applied to `risk_score`, which is the field a queue would sort on.

In [ ]:
tie = pd.DataFrame(res['tie_diagnostics']).T
tie['distinct'] = tie['distinct'].astype(int); tie['of_n'] = tie['of_n'].astype(int)
tie['distinct_per_1000_rows'] = (tie['distinct'] / tie['of_n'] * 1000).round(0)
display(tie)
print("`precision_at_k` returns the prevalence when the k-th rank is tied — which is why several")
print("arms in section 3 report exactly the base rate. That is the metric refusing to score an")
print("arbitrary selection, not a coincidence.")

## 7. The equity trade-off, measured rather than argued

`docs/model-design.md` §9.4 left the ranking-target choice open, on the correct grounds that it is a
product and ethical decision rather than a modelling one. What was missing was the price list. Here
it is, against a real outcome.

In [ ]:
pop = res['mean_vulnerability_all']
pick = ['expected_loss', 'expected_loss_log_exposure', 'risk_score', 'exposure_only']
fig, ax = plt.subplots(figsize=(7.5, 4.6))
for name in pick:
    e = res['rankings'][name]
    ax.scatter(e['auc'], e['mean_vulnerability_top50'], s=90,
               color=RED if name == 'expected_loss' else BLUE, zorder=3)
    ax.annotate(name, (e['auc'], e['mean_vulnerability_top50']),
                textcoords='offset points', xytext=(8, 4), fontsize=9)
ax.axhline(pop, color='k', ls='--', lw=1, label=f'population mean vulnerability = {pop}')
ax.axvline(0.5, color=GREY, ls=':', lw=1, label='chance AUC')
ax.set_xlabel('AUC against realised crop loss'); ax.set_ylabel('mean vulnerability of the top 50')
ax.set_title('Above the dashed line the queue prioritises the vulnerable; below it, the opposite')
ax.legend(fontsize=8, loc='lower left')
fig.tight_layout(); plt.show()

Read the quadrants literally:

* **`expected_loss`** — the production queue — sits **below** the population mean (0.728 vs 0.780).
  It puts *less* vulnerable farmers first, by arithmetic, exactly as §9.4 predicted.
* **`risk_score`** reverses that completely (0.954) — and ranks **worse than chance** against
  realised loss. Equity bought at the cost of finding anyone.
* **Log-compressing exposure** is the middle: it costs 0.025 AUC and buys +0.09 vulnerability.

That is the trade the product owes a decision on. It is no longer hypothetical, and nobody has to
guess at the cost of choosing fairly.

## 8. Decision curve — is acting on this better than the trivial strategies?

Net benefit (Vickers & Elkin 2006), the repo's own `lab.eval.evaluate.net_benefit`. `t` is the
probability at which a visit becomes worthwhile, so `t/(1-t)` is the missed-loss-to-wasted-trip cost
ratio. **The ceiling is the event rate, 0.2894.** A model is worth deploying only where it clears
*both* "visit everyone" and "visit nobody".

The composed score is not a probability of this label, so it is Platt-scaled **out of fold by EA**.
Fitting that scaling in-sample would let the curve report its own training fit. The bridge is an
assumption, stated here as `evaluate.prob_event` states its normal-CDF equivalent.

In [ ]:
ts = [c['threshold'] for c in res['rankings']['expected_loss']['decision_curve']]
fig, ax = plt.subplots(figsize=(8.5, 4.6))
for name, colr, lw in [('expected_loss', RED, 2.4), ('exposure_only', BLUE, 1.6),
                       ('E10_fitted_model__CEILING', '#2e7d32', 2.4)]:
    ax.plot(ts, [c['model'] for c in res['rankings'][name]['decision_curve']],
            'o-', color=colr, lw=lw, label=name, ms=4)
ax.plot(ts, [c['visit_all'] for c in res['rankings']['expected_loss']['decision_curve']],
        's--', color=GREY, lw=1.4, label='visit everyone', ms=4)
ax.axhline(0, color='k', lw=1, label='visit nobody')
ax.set_ylim(-0.08, 0.24)
ax.set_xlabel('threshold probability t   (cost ratio = t / (1 - t))')
ax.set_ylabel('net benefit  (ceiling = prevalence 0.2894)')
ax.set_title('Only the fitted model clears both trivial strategies across the range')
ax.legend(fontsize=8); fig.tight_layout(); plt.show()

curve = pd.DataFrame({n: [c['model'] for c in e['decision_curve']]
                      for n, e in res['rankings'].items()}, index=ts)
curve['visit_all'] = [c['visit_all'] for c in res['rankings']['expected_loss']['decision_curve']]
display(curve.round(4))

At `t <= 0.20` the composed equation is **identical to visiting everyone** — the scaled probabilities
put the whole population above the threshold, so it flags all of them and adds nothing. At `t = 0.30`
it clears "visit nobody" by 0.0093, which is to say barely. Only the fitted model clears both
strategies across the range.

## 9. What follows

1. **Do not ship a numbered queue.** §5 is decisive and independent of everything else here.
   Unordered top-k, or fix area measurement first.
2. **Fix `risk_score`'s rounding** (§6). One line, and §9.3a already set the precedent.
3. **The composition needs a reason to exist.** §3 says multiplying the terms costs ordering
   information. Either the multiplicative form is wrong *for ranking*, or hazard must be real before
   the product can be judged — and only outcome capture settles that.
4. **The fitted model beats the equation by +0.115 AUC** on the one outcome either has been scored
   against. That is the argument for spending the next quarter on outcome capture rather than on
   more composition.

**What this does not say.** It does not say the risk equation is wrong. Hazard is drawn here, the
decomposition remains separately defensible, and an extension officer can still act on *"water
deficit 60 mm at flowering, dominant hazard drought"* without trusting the composed rank at all.
The finding is narrower and firmer than that: **as a ranking key, the composed score does not
currently beat its own exposure term, and its order does not survive the measurement error this
repository has already documented.**

## Sources

* Vickers, A.J., Elkin, E.B. (2006). Decision curve analysis. *Medical Decision Making*
  26:565–574. **[verified]** — `docs/CONCEPTS.md` Part 6, implemented in `lab/eval/evaluate.py`.
* IPCC AR5/AR6 risk framing, `Risk = f(Hazard, Exposure, Vulnerability)`. **[verified]** as a body
  of work — `docs/CONCEPTS.md` Part 3.
* Nigeria GHS-Panel Wave 5, World Bank LSMS-ISA,
  [catalog/6410](https://microdata.worldbank.org/index.php/catalog/6410). Data-use agreement permits
  analysis, not redistribution; `data/lsms/` is git-ignored.
* Efron (1979), the bootstrap. **[UNVERIFIED]** — the cluster-resampling argument is this repo's own
  (`docs/CONCEPTS.md` Part 6, "why resample sites and never rows").

**Not citable, by design.** The AUCs, paired contrasts, rank-retention shares, tie counts and the
confound decomposition are **measurements on this repository's own data**. Nobody published them, so
a citation would be wrong. Their warrant is reproducibility: git SHA, seed and input files are
recorded in `e11_result.json`, and this notebook re-derives its figures from that artifact rather
than restating them. See `docs/CONCEPTS.md`, "On warrants, and what 'no source' means".